**Análisis de datos para la optimización y priorización del abastecimiento de medicamentos y dispositivos médicos en un hospital de alta complejidad, a partir de datos históricos de 2024 a mayo de 2026.**

1. Caracterizar el comportamiento histórico del consumo de medicamentos y dispositivos médicos por servicio farmacéutico, identificando patrones de rotación, variabilidad y comportamiento temporal de la demanda durante el período de análisis.

*Extracción e identificación de las fuentes de información*

In [ ]:
#importar librerias

import pandas as pd
import numpy as np
import zipfile
import os


# **AÑO 2024**

In [ ]:
#Cargue de base de datos 2024

from google.colab import files

archivo_subido = files.upload()

In [ ]:
# Verificar los archivos contenidos en el ZIP de 2024

ruta_zip = "2024.zip"

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    archivos = [archivo for archivo in archivo_zip.namelist()if archivo.lower().endswith(".xlsx")]

print("ARCHIVOS DE CONSUMO ENCONTRADOS")
print("=" * 50)

for archivo in sorted(archivos):
    print(archivo)

print("\nTotal de archivos Excel:", len(archivos))

ARCHIVOS DE CONSUMO ENCONTRADOS
2024/Salida Abril 2024.xlsx
2024/Salida Agosto.xlsx
2024/Salida Diciembre.xlsx
2024/Salida Enero 2024.xlsx
2024/Salida Febrero 2024.xlsx
2024/Salida Julio 2024.xlsx
2024/Salida Junio.xlsx
2024/Salida Marzo 2024.xlsx
2024/Salida Mayo.xlsx
2024/Salida Noviembre.xlsx
2024/Salida Octubre 2024.xlsx
2024/Salida Septiembre 2024.xlsx

Total de archivos Excel: 12


In [ ]:
# Revisar la estructura de columnas de cada archivo de 2024

estructuras = []

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    for archivo in sorted(archivos):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f, nrows=5)

        estructuras.append({
            "Archivo": archivo.split("/")[-1],
            "Numero_columnas": len(df.columns),
            "Columnas": " | ".join(df.columns.astype(str))})

estructura_2024 = pd.DataFrame(estructuras)

estructura_2024

In [ ]:
# Contar los registros de cada base mensual de 2024

registros_2024 = []

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    for archivo in sorted(archivos):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        registros_2024.append({
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df)
        })

registros_2024 = pd.DataFrame(registros_2024)

registros_2024

,Archivo,Registros
0,Salida Abril 2024.xlsx,540043
1,Salida Agosto.xlsx,527695
2,Salida Diciembre.xlsx,402199
3,Salida Enero 2024.xlsx,566539
4,Salida Febrero 2024.xlsx,536706
5,Salida Julio 2024.xlsx,559870
6,Salida Junio.xlsx,580092
7,Salida Marzo 2024.xlsx,520466
8,Salida Mayo.xlsx,448056
9,Salida Noviembre.xlsx,305283


In [ ]:
# Organizar los registros de 2024 en orden cronológico

orden_meses = {
    "Enero": 1,
    "Febrero": 2,
    "Marzo": 3,
    "Abril": 4,
    "Mayo": 5,
    "Junio": 6,
    "Julio": 7,
    "Agosto": 8,
    "Septiembre": 9,
    "Octubre": 10,
    "Noviembre": 11,
    "Diciembre": 12
}

def identificar_mes(nombre_archivo):
    for mes in orden_meses:
        if mes.lower() in nombre_archivo.lower():
            return mes
    return "No identificado"

registros_2024["Mes"] = registros_2024["Archivo"].apply(identificar_mes)

registros_2024["Mes_numero"] = registros_2024["Mes"].map(orden_meses)

registros_2024 = registros_2024.sort_values(
    "Mes_numero"
).reset_index(drop=True)

registros_2024

,Archivo,Registros,Mes,Mes_numero
0,Salida Enero 2024.xlsx,566539,Enero,1
1,Salida Febrero 2024.xlsx,536706,Febrero,2
2,Salida Marzo 2024.xlsx,520466,Marzo,3
3,Salida Abril 2024.xlsx,540043,Abril,4
4,Salida Mayo.xlsx,448056,Mayo,5
5,Salida Junio.xlsx,580092,Junio,6
6,Salida Julio 2024.xlsx,559870,Julio,7
7,Salida Agosto.xlsx,527695,Agosto,8
8,Salida Septiembre 2024.xlsx,498777,Septiembre,9
9,Salida Octubre 2024.xlsx,300417,Octubre,10


In [ ]:
# Calcular el total de registros de 2024

total_registros_2024 = registros_2024["Registros"].sum()

print("Total de registros de consumo en 2024:", total_registros_2024)

Total de registros de consumo en 2024: 5786143


*Validación inicial de la calidad de los datos*

In [ ]:
# VALIDACIÓN DE CALIDAD DE LAS BASES DE CONSUMO - 2024


meses_validos = {"ENERO", "FEBRERO", "MARZO", "ABRIL","MAYO", "JUNIO", "JULIO", "AGOSTO","SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"}

validacion_2024 = []

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    for archivo in sorted(archivos):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        # Normalizar nombres de columnas
        df.columns = (df.columns.astype(str).str.strip().str.upper())

        # Normalizar variables de texto
        df["MES"] = (df["MES"].astype(str).str.strip().str.upper())

        # Convertir cantidad a numérico
        df["CANTIDAD"] = pd.to_numeric(df["CANTIDAD"],errors="coerce")

        resultado = {
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df),
            "Nulos_Codigo": df["CODIGO_PROPIO"].isna().sum(),
            "Nulos_Medicamento": df["DES_MEDICAMENTO"].isna().sum(),
            "Nulos_Cantidad": df["CANTIDAD"].isna().sum(),
            "Nulos_Bodega": df["BODEGA"].isna().sum(),
            "Nulos_Des_UF": df["DES_UF"].isna().sum(),
            "Nulos_Mes": df["MES"].isna().sum(),
            "Nulos_Año": df["AÑO"].isna().sum(),
            "Cantidad_Cero": (df["CANTIDAD"] == 0).sum(),
            "Cantidad_Negativa": (df["CANTIDAD"] < 0).sum(),
            "Año_Diferente_2024": (df["AÑO"] != 2024).sum(),
            "Mes_No_Valido": (~df["MES"].isin(meses_validos)).sum()}

        validacion_2024.append(resultado)

validacion_2024 = pd.DataFrame(validacion_2024)

validacion_2024

,Archivo,Registros,Nulos_Codigo,Nulos_Medicamento,Nulos_Cantidad,Nulos_Bodega,Nulos_Des_UF,Nulos_Mes,Nulos_Año,Cantidad_Cero,Cantidad_Negativa,Año_Diferente_2024,Mes_No_Valido
0,Salida Abril 2024.xlsx,540043,0,0,0,0,0,0,0,0,0,0,0
1,Salida Agosto.xlsx,527695,0,0,0,0,0,0,0,0,0,0,0
2,Salida Diciembre.xlsx,402199,0,0,0,0,0,0,0,0,0,0,0
3,Salida Enero 2024.xlsx,566539,0,0,0,0,0,0,0,0,0,0,0
4,Salida Febrero 2024.xlsx,536706,0,0,0,0,0,0,0,0,0,0,0
5,Salida Julio 2024.xlsx,559870,0,0,0,0,0,0,0,0,0,0,0
6,Salida Junio.xlsx,580092,0,0,0,0,0,0,0,0,0,0,0
7,Salida Marzo 2024.xlsx,520466,0,0,0,0,0,0,0,0,0,0,0
8,Salida Mayo.xlsx,448056,0,0,0,0,0,0,0,0,0,0,0
9,Salida Noviembre.xlsx,305283,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Resumen general de la validación de calidad de 2024

columnas_control = [
    "Nulos_Codigo",
    "Nulos_Medicamento",
    "Nulos_Cantidad",
    "Nulos_Bodega",
    "Nulos_Des_UF",
    "Nulos_Mes",
    "Nulos_Año",
    "Cantidad_Cero",
    "Cantidad_Negativa",
    "Año_Diferente_2024",
    "Mes_No_Valido"
]

resumen_calidad_2024 = pd.DataFrame({
    "Control": columnas_control,
    "Total_registros_con_hallazgo": [
        validacion_2024[col].sum()
        for col in columnas_control
    ]
})

resumen_calidad_2024

,Control,Total_registros_con_hallazgo
0,Nulos_Codigo,0
1,Nulos_Medicamento,0
2,Nulos_Cantidad,0
3,Nulos_Bodega,0
4,Nulos_Des_UF,0
5,Nulos_Mes,0
6,Nulos_Año,0
7,Cantidad_Cero,0
8,Cantidad_Negativa,0
9,Año_Diferente_2024,0


*Caracterización inicial de productos, bodegas y unidades funcionales*

In [ ]:
# CARACTERIZACIÓN DE PRODUCTOS, BODEGAS Y UNIDADES FUNCIONALES BASES DE CONSUMO 2024


caracterizacion_2024 = []

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    for archivo in sorted(archivos):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        # Normalizar nombres de columnas
        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.upper()
        )

        caracterizacion_2024.append({
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df),
            "Codigos_producto": df["CODIGO_PROPIO"].nunique(),
            "Medicamentos_productos": df["DES_MEDICAMENTO"].nunique(),
            "Bodegas": df["BODEGA"].nunique(),
            "Unidades_funcionales": df["DES_UF"].nunique()
        })

caracterizacion_2024 = pd.DataFrame(caracterizacion_2024)

caracterizacion_2024

,Archivo,Registros,Codigos_producto,Medicamentos_productos,Bodegas,Unidades_funcionales
0,Salida Abril 2024.xlsx,540043,2617,2619,17,81
1,Salida Agosto.xlsx,527695,2668,2685,17,80
2,Salida Diciembre.xlsx,402199,2433,2439,19,81
3,Salida Enero 2024.xlsx,566539,2813,2811,17,117
4,Salida Febrero 2024.xlsx,536706,2690,2686,17,119
5,Salida Julio 2024.xlsx,559870,2629,2694,17,83
6,Salida Junio.xlsx,580092,2690,2689,17,82
7,Salida Marzo 2024.xlsx,520466,2626,2624,17,80
8,Salida Mayo.xlsx,448056,2562,2563,17,87
9,Salida Noviembre.xlsx,305283,2285,2290,19,82


In [ ]:
# Consolidar las variables necesarias para caracterizar el universo de 2024

datos_caracterizacion_2024 = []

with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:

    for archivo in sorted(archivos):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        df.columns = (df.columns.astype(str).str.strip().str.upper())

        datos_caracterizacion_2024.append(
            df[["CODIGO_PROPIO","DES_MEDICAMENTO","BODEGA","DES_UF"]])

universo_2024 = pd.concat(datos_caracterizacion_2024,ignore_index=True)

print("Registros analizados:", len(universo_2024))

Registros analizados: 5786143


In [ ]:
# Indicadores globales del universo de consumo 2024

total_codigos = universo_2024["CODIGO_PROPIO"].nunique()
total_productos = universo_2024["DES_MEDICAMENTO"].nunique()
total_bodegas = universo_2024["BODEGA"].nunique()
total_uf = universo_2024["DES_UF"].nunique()

print("CARACTERIZACIÓN GLOBAL 2024")
print("=" * 40)

print("Códigos de producto únicos:", total_codigos)
print("Descripciones de producto únicas:", total_productos)
print("Bodegas únicas:", total_bodegas)
print("Unidades funcionales únicas:", total_uf)

CARACTERIZACIÓN GLOBAL 2024
Códigos de producto únicos: 6732
Descripciones de producto únicas: 7009
Bodegas únicas: 19
Unidades funcionales únicas: 173


In [ ]:
# Identificar productos asociados a más de una bodega

producto_bodega = (
    universo_2024
    .groupby("CODIGO_PROPIO")["BODEGA"]
    .nunique()
)

productos_varias_bodegas = (
    producto_bodega > 1
).sum()

print(
    "Productos presentes en más de una bodega:",
    productos_varias_bodegas
)

Productos presentes en más de una bodega: 1908


# **AÑO 2025**

In [ ]:
from google.colab import files

archivo_subido_2025 = files.upload()

Saving 2025.zip to 2025.zip


In [ ]:
# Verificar los archivos contenidos en el ZIP de 2025

import zipfile

ruta_zip_2025 = "2025.zip"

with zipfile.ZipFile(ruta_zip_2025, "r") as archivo_zip:
    archivos_2025 = [
        archivo
        for archivo in archivo_zip.namelist()
        if archivo.lower().endswith(".xlsx")
    ]

print("ARCHIVOS DE CONSUMO ENCONTRADOS - 2025")
print("=" * 50)

for archivo in sorted(archivos_2025):
    print(archivo)

print("\nTotal de archivos Excel:", len(archivos_2025))

ARCHIVOS DE CONSUMO ENCONTRADOS - 2025
2025/Salida Abril 2025.xlsx
2025/Salida Agosto 2025.xlsx
2025/Salida Diciembre 2025.xlsx
2025/Salida Enero 2025.xlsx
2025/Salida Febrero 2025.xlsx
2025/Salida Junio 2025.xlsx
2025/Salida Marzo 2025.xlsx
2025/Salida Mayo 2025.xlsx
2025/Salida Noviembre 2025.xlsx
2025/Salida Octubre 2025.xlsx
2025/Salida Septiembre 2025.xlsx
2025/Salida julio 2025.xlsx

Total de archivos Excel: 12


In [ ]:
# Revisar la estructura de columnas de cada archivo de 2025

estructuras_2025 = []

with zipfile.ZipFile(ruta_zip_2025, "r") as archivo_zip:
    for archivo in sorted(archivos_2025):
        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f, nrows=5)

        estructuras_2025.append({
            "Archivo": archivo.split("/")[-1],
            "Numero_columnas": len(df.columns),
            "Columnas": " | ".join(df.columns.astype(str))
        })

estructura_2025 = pd.DataFrame(estructuras_2025)

pd.set_option("display.max_colwidth", None)

estructura_2025[
    ["Archivo", "Numero_columnas", "Columnas"]
]

,Archivo,Numero_columnas,Columnas
0,Salida Abril 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
1,Salida Agosto 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | BODEGA | CANTIDAD | DES_UF | MES | AÑO
2,Salida Diciembre 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | BODEGA | CANTIDAD | DES_UF | MES | AÑO
3,Salida Enero 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
4,Salida Febrero 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
5,Salida Junio 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
6,Salida Marzo 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | BODEGA | CANTIDAD | DES_UF | MES | AÑO
7,Salida Mayo 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
8,Salida Noviembre 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | mes | año
9,Salida Octubre 2025.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | mes | año


In [ ]:
# Contar los registros de cada base mensual de 2025

registros_2025 = []

with zipfile.ZipFile(ruta_zip_2025, "r") as archivo_zip:
    for archivo in sorted(archivos_2025):
        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        registros_2025.append({
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df)
        })

registros_2025 = pd.DataFrame(registros_2025)

registros_2025

,Archivo,Registros
0,Salida Abril 2025.xlsx,423454
1,Salida Agosto 2025.xlsx,317065
2,Salida Diciembre 2025.xlsx,316572
3,Salida Enero 2025.xlsx,419858
4,Salida Febrero 2025.xlsx,335108
5,Salida Junio 2025.xlsx,330515
6,Salida Marzo 2025.xlsx,374760
7,Salida Mayo 2025.xlsx,333918
8,Salida Noviembre 2025.xlsx,371316
9,Salida Octubre 2025.xlsx,395075


In [ ]:
# Organizar los registros de 2025 en orden cronológico

orden_meses = {
    "Enero": 1,
    "Febrero": 2,
    "Marzo": 3,
    "Abril": 4,
    "Mayo": 5,
    "Junio": 6,
    "Julio": 7,
    "Agosto": 8,
    "Septiembre": 9,
    "Octubre": 10,
    "Noviembre": 11,
    "Diciembre": 12
}

def identificar_mes(nombre_archivo):
    for mes in orden_meses:
        if mes.lower() in nombre_archivo.lower():
            return mes
    return "No identificado"

registros_2025["Mes"] = (
    registros_2025["Archivo"]
    .apply(identificar_mes)
)

registros_2025["Mes_numero"] = (
    registros_2025["Mes"]
    .map(orden_meses)
)

registros_2025 = (
    registros_2025
    .sort_values("Mes_numero")
    .reset_index(drop=True)
)

registros_2025

,Archivo,Registros,Mes,Mes_numero
0,Salida Enero 2025.xlsx,419858,Enero,1
1,Salida Febrero 2025.xlsx,335108,Febrero,2
2,Salida Marzo 2025.xlsx,374760,Marzo,3
3,Salida Abril 2025.xlsx,423454,Abril,4
4,Salida Mayo 2025.xlsx,333918,Mayo,5
5,Salida Junio 2025.xlsx,330515,Junio,6
6,Salida julio 2025.xlsx,386709,Julio,7
7,Salida Agosto 2025.xlsx,317065,Agosto,8
8,Salida Septiembre 2025.xlsx,384461,Septiembre,9
9,Salida Octubre 2025.xlsx,395075,Octubre,10


In [ ]:
# Calcular el total de registros de 2025

total_registros_2025 = registros_2025["Registros"].sum()

print(
    "Total de registros de consumo en 2025:",
    total_registros_2025
)

Total de registros de consumo en 2025: 4388811


In [ ]:
# Validación de calidad de las bases de consumo - 2025

meses_validos = {
    "ENERO", "FEBRERO", "MARZO", "ABRIL",
    "MAYO", "JUNIO", "JULIO", "AGOSTO",
    "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
}

validacion_2025 = []

with zipfile.ZipFile(ruta_zip_2025, "r") as archivo_zip:
    for archivo in sorted(archivos_2025):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df["MES"] = (
            df["MES"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df["CANTIDAD"] = pd.to_numeric(
            df["CANTIDAD"],
            errors="coerce"
        )

        resultado = {
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df),
            "Nulos_Codigo": df["CODIGO_PROPIO"].isna().sum(),
            "Nulos_Medicamento": df["DES_MEDICAMENTO"].isna().sum(),
            "Nulos_Cantidad": df["CANTIDAD"].isna().sum(),
            "Nulos_Bodega": df["BODEGA"].isna().sum(),
            "Nulos_Des_UF": df["DES_UF"].isna().sum(),
            "Nulos_Mes": df["MES"].isna().sum(),
            "Nulos_Año": df["AÑO"].isna().sum(),
            "Cantidad_Cero": (df["CANTIDAD"] == 0).sum(),
            "Cantidad_Negativa": (df["CANTIDAD"] < 0).sum(),
            "Año_Diferente_2025": (df["AÑO"] != 2025).sum(),
            "Mes_No_Valido": (~df["MES"].isin(meses_validos)).sum()
        }

        validacion_2025.append(resultado)

validacion_2025 = pd.DataFrame(validacion_2025)

validacion_2025

,Archivo,Registros,Nulos_Codigo,Nulos_Medicamento,Nulos_Cantidad,Nulos_Bodega,Nulos_Des_UF,Nulos_Mes,Nulos_Año,Cantidad_Cero,Cantidad_Negativa,Año_Diferente_2025,Mes_No_Valido
0,Salida Abril 2025.xlsx,423454,0,0,0,0,0,0,0,0,0,0,0
1,Salida Agosto 2025.xlsx,317065,0,0,0,0,0,0,0,0,4,0,0
2,Salida Diciembre 2025.xlsx,316572,0,0,0,0,0,0,0,0,0,0,0
3,Salida Enero 2025.xlsx,419858,0,0,0,0,0,0,0,0,0,0,0
4,Salida Febrero 2025.xlsx,335108,0,0,0,0,0,0,0,0,0,0,0
5,Salida Junio 2025.xlsx,330515,0,0,0,0,0,0,0,0,0,0,0
6,Salida Marzo 2025.xlsx,374760,0,0,0,0,0,0,0,0,0,0,0
7,Salida Mayo 2025.xlsx,333918,0,0,1,0,0,0,0,0,0,0,0
8,Salida Noviembre 2025.xlsx,371316,0,0,0,0,0,0,0,0,9,0,0
9,Salida Octubre 2025.xlsx,395075,0,0,0,0,0,0,0,0,4,0,0


In [ ]:
# Resumen general de la validación de calidad de 2025

columnas_control = [
    "Nulos_Codigo",
    "Nulos_Medicamento",
    "Nulos_Cantidad",
    "Nulos_Bodega",
    "Nulos_Des_UF",
    "Nulos_Mes",
    "Nulos_Año",
    "Cantidad_Cero",
    "Cantidad_Negativa",
    "Año_Diferente_2025",
    "Mes_No_Valido"
]

resumen_calidad_2025 = pd.DataFrame({
    "Control": columnas_control,
    "Total_registros_con_hallazgo": [
        validacion_2025[col].sum()
        for col in columnas_control
    ]
})

resumen_calidad_2025

,Control,Total_registros_con_hallazgo
0,Nulos_Codigo,0
1,Nulos_Medicamento,0
2,Nulos_Cantidad,1
3,Nulos_Bodega,0
4,Nulos_Des_UF,0
5,Nulos_Mes,0
6,Nulos_Año,0
7,Cantidad_Cero,0
8,Cantidad_Negativa,37
9,Año_Diferente_2025,0


In [ ]:
# Caracterización de productos, bodegas y unidades funcionales bases de consumo 2025

datos_caracterizacion_2025 = []

with zipfile.ZipFile(ruta_zip_2025, "r") as archivo_zip:
    for archivo in sorted(archivos_2025):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.upper()
        )

        datos_caracterizacion_2025.append(
            df[
                [
                    "CODIGO_PROPIO",
                    "DES_MEDICAMENTO",
                    "BODEGA",
                    "DES_UF"
                ]
            ]
        )

universo_2025 = pd.concat(
    datos_caracterizacion_2025,
    ignore_index=True
)

print(
    "Registros analizados:",
    len(universo_2025)
)

Registros analizados: 4388811


In [ ]:
# Indicadores globales del universo de consumo 2025

total_codigos_2025 = (
    universo_2025["CODIGO_PROPIO"].nunique()
)

total_productos_2025 = (
    universo_2025["DES_MEDICAMENTO"].nunique()
)

total_bodegas_2025 = (
    universo_2025["BODEGA"].nunique()
)

total_uf_2025 = (
    universo_2025["DES_UF"].nunique()
)

print("CARACTERIZACIÓN GLOBAL 2025")
print("=" * 40)

print(
    "Códigos de producto únicos:",
    total_codigos_2025
)

print(
    "Descripciones de producto únicas:",
    total_productos_2025
)

print(
    "Bodegas únicas:",
    total_bodegas_2025
)

print(
    "Unidades funcionales únicas:",
    total_uf_2025
)

CARACTERIZACIÓN GLOBAL 2025
Códigos de producto únicos: 8506
Descripciones de producto únicas: 7247
Bodegas únicas: 25
Unidades funcionales únicas: 126


In [ ]:
# Identificar productos asociados a más de una bodega 2025

producto_bodega_2025 = (
    universo_2025
    .groupby("CODIGO_PROPIO")["BODEGA"]
    .nunique()
)

productos_varias_bodegas_2025 = (
    producto_bodega_2025 > 1
).sum()

print(
    "Productos presentes en más de una bodega:",
    productos_varias_bodegas_2025
)

Productos presentes en más de una bodega: 2196


# **AÑO 2026**

In [ ]:
from google.colab import files

archivo_subido_2026 = files.upload()

Saving 2026.zip to 2026.zip


In [ ]:
import zipfile

ruta_zip_2026 = "2026.zip"

with zipfile.ZipFile(ruta_zip_2026, "r") as archivo_zip:
    archivos_2026 = [
        archivo
        for archivo in archivo_zip.namelist()
        if archivo.lower().endswith(".xlsx")
    ]

print("ARCHIVOS DE CONSUMO ENCONTRADOS - 2026")
print("=" * 50)

for archivo in sorted(archivos_2026):
    print(archivo)

print("\nTotal de archivos Excel:", len(archivos_2026))

ARCHIVOS DE CONSUMO ENCONTRADOS - 2026
2026/Salida Abril2026.xlsx
2026/Salida Enero 2026.xlsx
2026/Salida Febrero 2026.xlsx
2026/Salida Marzo 2026.xlsx
2026/Salida Mayo 2026.xlsx

Total de archivos Excel: 5


In [ ]:
estructuras_2026 = []

with zipfile.ZipFile(ruta_zip_2026, "r") as archivo_zip:
    for archivo in sorted(archivos_2026):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f, nrows=5)

        estructuras_2026.append({
            "Archivo": archivo.split("/")[-1],
            "Numero_columnas": len(df.columns),
            "Columnas": " | ".join(df.columns.astype(str))
        })

estructura_2026 = pd.DataFrame(estructuras_2026)

pd.set_option("display.max_colwidth", None)

estructura_2026[
    ["Archivo", "Numero_columnas", "Columnas"]
]

,Archivo,Numero_columnas,Columnas
0,Salida Abril2026.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
1,Salida Enero 2026.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | BODEGA | CANTIDAD | DES_UF | MES | AÑO
2,Salida Febrero 2026.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | BODEGA | CANTIDAD | DES_UF | MES | AÑO
3,Salida Marzo 2026.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | MES | AÑO
4,Salida Mayo 2026.xlsx,7,CODIGO_PROPIO | DES_MEDICAMENTO | CANTIDAD | BODEGA | DES_UF | mes | año


In [ ]:
registros_2026 = []

with zipfile.ZipFile(ruta_zip_2026, "r") as archivo_zip:

    for archivo in sorted(archivos_2026):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        registros_2026.append({
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df)
        })

registros_2026 = pd.DataFrame(registros_2026)

registros_2026

,Archivo,Registros
0,Salida Abril2026.xlsx,280318
1,Salida Enero 2026.xlsx,310310
2,Salida Febrero 2026.xlsx,293350
3,Salida Marzo 2026.xlsx,322600
4,Salida Mayo 2026.xlsx,243929


In [ ]:
orden_meses_2026 = {
    "Enero": 1,
    "Febrero": 2,
    "Marzo": 3,
    "Abril": 4,
    "Mayo": 5
}

def identificar_mes_2026(nombre_archivo):
    for mes in orden_meses_2026:
        if mes.lower() in nombre_archivo.lower():
            return mes
    return "No identificado"

registros_2026["Mes"] = (
    registros_2026["Archivo"]
    .apply(identificar_mes_2026)
)

registros_2026["Mes_numero"] = (
    registros_2026["Mes"]
    .map(orden_meses_2026)
)

registros_2026 = (
    registros_2026
    .sort_values("Mes_numero")
    .reset_index(drop=True)
)

registros_2026

,Archivo,Registros,Mes,Mes_numero
0,Salida Enero 2026.xlsx,310310,Enero,1
1,Salida Febrero 2026.xlsx,293350,Febrero,2
2,Salida Marzo 2026.xlsx,322600,Marzo,3
3,Salida Abril2026.xlsx,280318,Abril,4
4,Salida Mayo 2026.xlsx,243929,Mayo,5


In [ ]:
total_registros_2026 = registros_2026["Registros"].sum()

print(
    "Total de registros de consumo enero-mayo 2026:",
    total_registros_2026
)

Total de registros de consumo enero-mayo 2026: 1450507


In [ ]:
meses_validos_2026 = {
    "ENERO",
    "FEBRERO",
    "MARZO",
    "ABRIL",
    "MAYO"
}

validacion_2026 = []

with zipfile.ZipFile(ruta_zip_2026, "r") as archivo_zip:

    for archivo in sorted(archivos_2026):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df["MES"] = (
            df["MES"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        df["CANTIDAD"] = pd.to_numeric(
            df["CANTIDAD"],
            errors="coerce"
        )

        resultado = {
            "Archivo": archivo.split("/")[-1],
            "Registros": len(df),

            "Nulos_Codigo":
                df["CODIGO_PROPIO"].isna().sum(),

            "Nulos_Medicamento":
                df["DES_MEDICAMENTO"].isna().sum(),

            "Nulos_Cantidad":
                df["CANTIDAD"].isna().sum(),

            "Nulos_Bodega":
                df["BODEGA"].isna().sum(),

            "Nulos_Des_UF":
                df["DES_UF"].isna().sum(),

            "Nulos_Mes":
                df["MES"].isna().sum(),

            "Nulos_Año":
                df["AÑO"].isna().sum(),

            "Cantidad_Cero":
                (df["CANTIDAD"] == 0).sum(),

            "Cantidad_Negativa":
                (df["CANTIDAD"] < 0).sum(),

            "Año_Diferente_2026":
                (df["AÑO"] != 2026).sum(),

            "Mes_Fuera_Periodo":
                (~df["MES"].isin(meses_validos_2026)).sum()
        }

        validacion_2026.append(resultado)

validacion_2026 = pd.DataFrame(validacion_2026)

validacion_2026

,Archivo,Registros,Nulos_Codigo,Nulos_Medicamento,Nulos_Cantidad,Nulos_Bodega,Nulos_Des_UF,Nulos_Mes,Nulos_Año,Cantidad_Cero,Cantidad_Negativa,Año_Diferente_2026,Mes_Fuera_Periodo
0,Salida Abril2026.xlsx,280318,0,0,0,0,0,0,0,0,5,0,0
1,Salida Enero 2026.xlsx,310310,0,0,0,0,0,0,0,0,0,0,0
2,Salida Febrero 2026.xlsx,293350,0,0,0,0,0,0,0,0,9,0,0
3,Salida Marzo 2026.xlsx,322600,0,0,0,0,0,0,0,0,10,0,0
4,Salida Mayo 2026.xlsx,243929,0,0,0,0,0,0,0,0,7,0,0


In [ ]:
columnas_control_2026 = [
    "Nulos_Codigo",
    "Nulos_Medicamento",
    "Nulos_Cantidad",
    "Nulos_Bodega",
    "Nulos_Des_UF",
    "Nulos_Mes",
    "Nulos_Año",
    "Cantidad_Cero",
    "Cantidad_Negativa",
    "Año_Diferente_2026",
    "Mes_Fuera_Periodo"
]

resumen_calidad_2026 = pd.DataFrame({
    "Control": columnas_control_2026,
    "Total_registros_con_hallazgo": [
        validacion_2026[col].sum()
        for col in columnas_control_2026
    ]
})

resumen_calidad_2026

,Control,Total_registros_con_hallazgo
0,Nulos_Codigo,0
1,Nulos_Medicamento,0
2,Nulos_Cantidad,0
3,Nulos_Bodega,0
4,Nulos_Des_UF,0
5,Nulos_Mes,0
6,Nulos_Año,0
7,Cantidad_Cero,0
8,Cantidad_Negativa,31
9,Año_Diferente_2026,0


In [ ]:
datos_caracterizacion_2026 = []

with zipfile.ZipFile(ruta_zip_2026, "r") as archivo_zip:

    for archivo in sorted(archivos_2026):

        with archivo_zip.open(archivo) as f:
            df = pd.read_excel(f)

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.upper()
        )

        datos_caracterizacion_2026.append(
            df[
                [
                    "CODIGO_PROPIO",
                    "DES_MEDICAMENTO",
                    "BODEGA",
                    "DES_UF"
                ]
            ]
        )

universo_2026 = pd.concat(
    datos_caracterizacion_2026,
    ignore_index=True
)

print(
    "Registros analizados:",
    len(universo_2026)
)

Registros analizados: 1450507


In [ ]:
total_codigos_2026 = (
    universo_2026["CODIGO_PROPIO"].nunique()
)

total_productos_2026 = (
    universo_2026["DES_MEDICAMENTO"].nunique()
)

total_bodegas_2026 = (
    universo_2026["BODEGA"].nunique()
)

total_uf_2026 = (
    universo_2026["DES_UF"].nunique()
)

print("CARACTERIZACIÓN GLOBAL ENERO-MAYO 2026")
print("=" * 50)

print(
    "Códigos de producto únicos:",
    total_codigos_2026
)

print(
    "Descripciones de producto únicas:",
    total_productos_2026
)

print(
    "Bodegas únicas:",
    total_bodegas_2026
)

print(
    "Unidades funcionales únicas:",
    total_uf_2026
)

CARACTERIZACIÓN GLOBAL ENERO-MAYO 2026
Códigos de producto únicos: 3971
Descripciones de producto únicas: 4019
Bodegas únicas: 25
Unidades funcionales únicas: 107


In [ ]:
producto_bodega_2026 = (
    universo_2026
    .groupby("CODIGO_PROPIO")["BODEGA"]
    .nunique()
)

productos_varias_bodegas_2026 = (
    producto_bodega_2026 > 1
).sum()

print(
    "Productos presentes en más de una bodega:",
    productos_varias_bodegas_2026
)

Productos presentes en más de una bodega: 1071
